# Importação de Pacotes

In [ ]:
#leitura da base de dados
import pandas as pd
from pathlib import Path
import parquet

#modelo preditivo escolhido
import catboost as cb
from catboost import CatBoostClassifier

#validação cruzada
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, HalvingGridSearchCV
import numpy as np

#vetorização
from langchain_experimental.text_splitter import SemanticChunker 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

#métricas
import matplotlib
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, classification_report

#pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer

In [ ]:
def estimadores(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)

    acuracia = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    roc_auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class='ovr',
        average='weighted'
    )

    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test)

    print(f"""
        Acurácia: {acuracia:.3f}
        Recall (weighted): {recall:.3f}
        F1-score (weighted): {f1:.3f}
        ROC AUC (ovr): {roc_auc:.3f}
        """)

    print(classification_report(y_test, y_pred))

## Leitura DataFrame

In [ ]:
df_erro_simples = pd.read_parquet("erro_medico_tidy_final.parquet")
df = df_erro_simples

### Escolha do Modelo

In [ ]:
modelo= CatBoostClassifier(auto_class_weights='Balanced')

### Escolha dos HiperParâmetros

In [ ]:
parametros = {
    'modelo__iterations': [300],
    'modelo__depth': [6],
    'modelo__learning_rate': [0.1],
    'modelo__verbose': [0]
}

In [ ]:
df.head(2)

In [ ]:
df.columns

### Preparação Embedding

In [ ]:
model_name = "rufimelo/bert-large-portuguese-cased-sts"

model_kwargs = {"device" : "cuda"}

encode_kwargs = {"normalize_embeddings" : False}

embeddings = HuggingFaceEmbeddings(
    model_name = model_name,
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

vector_store = InMemoryVectorStore(embeddings)
text_splitter = SemanticChunker(embeddings, breakpoint_threshold_type="gradient")

texts = text_splitter.create_documents(corpus)
document_ids = vector_store.add_documents(documents=texts)

obj_embed = embeddings.embed_documents(corpus)

df_embed = pd.DataFrame(obj_embed)


In [ ]:
#axis=1 concatena colunas
pd.concat([df, df_embed], axis= 1)

# Aplicação de Pipelines

In [ ]:
lista_X = ['valor_acao','n_autores', 'n_reus', 'tem_hospital', 'tem_plano_saude',
           'tem_ente_publico', 'tem_medico_individual', 'n_adv_autor', 'n_adv_reu',
           'tem_perito', 'tem_denuncia_lide', 'tem_assistente']

X = df[lista_X]

y = df["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)


pipeline = Pipeline([
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='f1_macro',
    refit= True,
    cv=5
)

searchCV_pipeline.fit(X_train, y_train)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)